# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


**Workspace maintenance:** Open `/home/sagemaker-user/NFL_WORKSPACE.ipynb` for current inventory. Saved outputs below are historical. Missing prerequisites are not model failures. Restart the kernel when changing kit folders. Do not run experiments until the workspace review is complete.

# 31 · Goal-aligned motion: matched feature ablation

**Prepared for later manual execution. Do not run before the readiness checkpoint has been reviewed.**

Three arms share the existing full-motion representation and all candidate masks. `mask` supplies zero new numerical values, `core` supplies six, and `full` supplies all twelve. This is an exploratory small-sample comparison, not an untouched evaluation. No active review release is included in the package.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import plotly.io as pio
KIT=Path('/home/sagemaker-user/nfl_feature_rounds14_15')
OUT=Path('/home/sagemaker-user/nfl-feature-round14-results')
PY=Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
ROUND=14
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space and upload/extract the package first.')
# A fresh kernel prevents cross-round modules from being reused silently.
_previous_kit = globals().get('_NFL_ACTIVE_KIT')
if _previous_kit is not None and _previous_kit != str(KIT):
    raise RuntimeError('Restart the kernel before switching NFL kit folders. No stage was started.')
for _module_name in [p.stem for p in KIT.glob('*.py')]:
    _module = sys.modules.get(_module_name)
    _location = getattr(_module, '__file__', None)
    if _location and 'nfl' in str(_location).lower() and Path(_location).resolve().parent != KIT.resolve():
        raise RuntimeError('A helper from another NFL round is cached. Restart the kernel; do not reinstall packages.')
_NFL_ACTIVE_KIT = str(KIT)
sys.path.insert(0,str(KIT))
import os
os.chdir(KIT)
import visuals
pio.renderers.default='plotly_mimetype'
PROCEED = False
def run(stage,*options):
    if stage in ('prepare','runtime','profile','train','evaluate','replay') and not PROCEED:
        print('PAUSED: required readiness or review is missing. No stage was executed. Open NFL_WORKSPACE.ipynb.')
        return False
    cmd=[str(PY),str(KIT/'run_round.py'),stage,'--round',str(ROUND),*options]
    with subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
        try:
            for line in p.stdout: print(line,end='')
            status=p.wait()
        except KeyboardInterrupt:
            p.send_signal(2)
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired: p.kill();p.wait()
            raise
    if status: raise RuntimeError(f'{stage} stopped ({status}). Export report; do not change settings or retry unchanged.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()


## Explicit readiness-review gate
A genuine manual decision must bind the two smoke reports, the label audit, and current test/source hashes. The template alone is insufficient. Do not fabricate a passed decision or bypass a stop.

In [ ]:
from types import SimpleNamespace
from readiness import require_review_release
PROCEED = False
try:
    release = require_review_release(SimpleNamespace(kit=KIT, out=OUT, round_no=ROUND))
except FileNotFoundError as exc:
    print('PREREQUISITE MISSING — no work started:', str(exc))
    print('Open /home/sagemaker-user/NFL_WORKSPACE.ipynb for the inventory; do not rerun training.')
except ValueError as exc:
    print('REVIEW OR INTEGRITY CHECK PENDING — no work started:', str(exc))
    print('Return nfl_workspace_report.zip; do not edit releases, signatures, or tolerances.')
else:
    PROCEED = True
    print('READY: the existing review matches these receipts. No stage has started yet.')
    print(json.dumps(release, indent=2))


## Prepare the complete feature population; validate cached CPU runtime
The code uses existing data and package caches. It will not install packages online. Runtime tests include the optimizer checkpoint interruption/continuation check.

In [ ]:
run('prepare')
run('runtime')

## Training-only budget gate
A failed profile remains stopped; repeating it does not draw another timing sample until one passes. Profile steps are disposable and are not counted as scientific fits.

In [ ]:
run('profile')

## Fixed-exposure models
Same architecture, initialization, data, batch order and optimizer exposure. No evaluation-based checkpoint selection. Run one cell at a time.

In [ ]:
run('train','--arm','mask')

In [ ]:
run('train','--arm','core')

In [ ]:
run('train','--arm','full')

## Complete-pair evaluation and diagnostics
The three planned contrasts use all forecast rows and pooled coordinate RMSE. The interval correction covers six contrasts across the two rounds, not every previous adaptive decision.

In [ ]:
if PROCEED:
    run('evaluate')
    show(visuals.learning(OUT),'learning_curves')
    show(visuals.metrics(OUT),'matched_metrics')
    show(visuals.contrasts(OUT),'paired_contrasts')
    show(visuals.horizons(OUT),'horizon_errors')
else:
    print('No evaluation or figures generated: prerequisites are pending.')


## Fresh-process replay and report
Replay refuses missing/incomplete models and may not fit them. Require zero new optimizer steps. Save, reopen, and inspect the four inline figures.

In [ ]:
run('replay')
run('report')
print(OUT/f'nfl_feature_round{ROUND}_report.zip')